In [1]:
from datasets import load_dataset

ds = load_dataset("syedkhalid0/Sentiment-Analysis")

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 83989
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 10499
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 10499
    })
})

In [3]:
train_df = ds["train"].to_pandas()
val_df   = ds["validation"].to_pandas()
test_df  = ds["test"].to_pandas()

In [4]:
train_df.head()

,text,label
0,almost got in a giant car accident on the 101,0
1,like something wholly original,2
2,b.s. one another,0
3,"Happy Star Wars Day, may the 4th be with you ...",2
4,few new converts,0


## Initializing the train and test set

In [5]:
import pandas as pd
train_df = train_df.sample(n=10000, random_state=42)
test_df = test_df.sample(n=2000, random_state=42)
train_df.head()

,text,label
1032,with an obvious rapport with her actors and a ...,2
75157,insightfully,2
75064,"elling '' manages to do all three quite well ,...",2
31698,leaves a lot to be desired .,0
7781,"sweet , charming",2


In [6]:
test_df.head()

,text,label
5118,This wasn't worth the price. I bought this as ...,0
7396,Very funny Thanks,2
8514,simply a re-hash of the other seven films,0
7281,therefore i know better than to rush to the th...,0
8649,from the characters,2


In [7]:
print(train_df.shape)
print(test_df.shape)

(10000, 2)
(2000, 2)


## Text preprocessing
Cleans the text for supervised learning using(HTML stripping, accent removal, contraction expansion, lowercasing, lemmatization, special-character removal, stopword removal).

Normalise an iterable of raw texts for supervised learning.

In [8]:
!pip install spacy
!pip install contractions

In [9]:
import nltk
import spacy
import pandas as pd
import numpy as np
from nltk.tokenize.toktok import ToktokTokenizer
import re
from bs4 import BeautifulSoup
import unicodedata

In [10]:
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     C:\Users\navee\AppData\Roaming\nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     C:\Users\navee\AppData\Roaming\nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     C:\Users\navee\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     C:\Users\navee\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     C:\Users\navee\AppData\Roaming\nltk_data...
[

True

In [13]:
from contractions import contractions_dict
nlp = spacy.load('en_core_web_sm')
tokenizer = ToktokTokenizer()
stopword_list = nltk.corpus.stopwords.words('english')
stopword_list.remove('no')
stopword_list.remove('not')


# # Cleaning Text - strip HTML
def strip_html_tags(text):
    soup = BeautifulSoup(text, "html.parser")
    stripped_text = soup.get_text()
    return stripped_text


# # Removing accented characters
def remove_accented_chars(text):
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    return text


# # Expanding Contractions
def expand_contractions(text, contraction_mapping=contractions_dict):

    contractions_pattern = re.compile('({})'.format('|'.join(contraction_mapping.keys())),
                                      flags=re.IGNORECASE|re.DOTALL)
    def expand_match(contraction):
        match = contraction.group(0)
        first_char = match[0]
        if contraction_mapping.get(match) is not  None :
            expanded_contraction = contraction_mapping.get(match) \
                                   if contraction_mapping.get(match) \
                                    else contraction_mapping.get(match.lower())
            expanded_contraction = first_char+expanded_contraction[1:]
            return expanded_contraction
        else: return match

    expanded_text = contractions_pattern.sub(expand_match, text)
    expanded_text = re.sub("'", "", expanded_text)
    return expanded_text


# # Removing Special Characters
def remove_special_characters(text):
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text


# # Lemmatizing text
def lemmatize_text(text):
    text = nlp(text)
    text = ' '.join([word.lemma_ if word.lemma_ != '-PRON-' else word.text for word in text])
    return text


# # Removing Stopwords
def remove_stopwords(text, is_lower_case=False):
    tokens = tokenizer.tokenize(text)
    tokens = [token.strip() for token in tokens]
    if is_lower_case:
        filtered_tokens = [token for token in tokens if token not in stopword_list]
    else:
        filtered_tokens = [token for token in tokens if token.lower() not in stopword_list]
    filtered_text = ' '.join(filtered_tokens)
    return filtered_text


# # Normalize text corpus - tying it all together
def normalize_corpus(corpus, html_stripping=True, contraction_expansion=True,
                     accented_char_removal=True, text_lower_case=True,
                     text_lemmatization=True, special_char_removal=True,
                     stopword_removal=True):

    normalized_corpus = []

    for doc in corpus:

        if html_stripping:
            doc = strip_html_tags(doc)

        if accented_char_removal:
            doc = remove_accented_chars(doc)

        if contraction_expansion:
            doc = expand_contractions(doc)

        if text_lower_case:
            doc = doc.lower()

        # remove extra newlines
        doc = re.sub(r'[\r|\n|\r\n]+', ' ',doc)
        # insert spaces between special characters to isolate them
        special_char_pattern = re.compile(r'([{.(-)!}])')
        doc = special_char_pattern.sub(" \\1 ", doc)

        if text_lemmatization:
            doc = lemmatize_text(doc)

        if special_char_removal:
            doc = remove_special_characters(doc)

        # remove extra whitespace
        doc = re.sub(' +', ' ', doc)

        if stopword_removal:
            doc = remove_stopwords(doc, is_lower_case=text_lower_case)

        normalized_corpus.append(doc)

    return normalized_corpus


In [14]:
def preprocess_text(texts):
    return normalize_corpus(
        list(texts),
        html_stripping=True,
        accented_char_removal=True,
        contraction_expansion=True,
        text_lower_case=True,
        text_lemmatization=True,
        special_char_removal=True,
        stopword_removal=True,
    )


# Keep the raw text and add a cleaned column
train_df["clean_text"] = preprocess_text(train_df["text"])
test_df["clean_text"] = preprocess_text(test_df["text"])

train_df[["text", "clean_text", "label"]].head()

,text,clean_text,label
1032,with an obvious rapport with her actors and a ...,obvious rapport actor striking style behind ca...,2
75157,insightfully,insightfully,2
75064,"elling '' manages to do all three quite well ,...",elle manage three quite well make one year enj...,2
31698,leaves a lot to be desired .,leave lot desire,0
7781,"sweet , charming",sweet charming,2


Next we will be using tf-idf for vectorising the cleaned text for ml models

In [15]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [16]:
from sklearn import metrics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone
from sklearn.preprocessing import label_binarize
from numpy import interp
from sklearn.metrics import roc_curve, auc


def get_metrics(true_labels, predicted_labels):

    print('Accuracy:', np.round(
                        metrics.accuracy_score(true_labels,
                                               predicted_labels),
                        4))
    print('Precision:', np.round(
                        metrics.precision_score(true_labels,
                                               predicted_labels,
                                               average='weighted'),
                        4))
    print('Recall:', np.round(
                        metrics.recall_score(true_labels,
                                               predicted_labels,
                                               average='weighted'),
                        4))
    print('F1 Score:', np.round(
                        metrics.f1_score(true_labels,
                                               predicted_labels,
                                               average='weighted'),
                        4))


def train_predict_model(classifier,
                        train_features, train_labels,
                        test_features, test_labels):
    # build model
    classifier.fit(train_features, train_labels)
    # predict using model
    predictions = classifier.predict(test_features)
    return predictions


def display_confusion_matrix(true_labels, predicted_labels, classes=[1,0]):

    total_classes = len(classes)
    level_labels = [total_classes*[0], list(range(total_classes))]
    cm = metrics.confusion_matrix(y_true=true_labels, y_pred=predicted_labels, labels=classes)
    cm_frame = pd.DataFrame(data=cm, columns=pd.MultiIndex(levels=[['Predicted:'], classes], codes=level_labels), index=pd.MultiIndex(levels=[['Actual:'], classes], codes=level_labels))
    print(cm_frame)


def display_confusion_matrix_pretty(true_labels, predicted_labels, classes=[1,0]):

    total_classes = len(classes)
    level_labels = [total_classes*[0], list(range(total_classes))]

    cm = metrics.confusion_matrix(y_true=true_labels, y_pred=predicted_labels, labels=classes)
    cm_frame = pd.DataFrame(data=cm, columns=pd.MultiIndex(levels=[['Predicted:'], classes], labels=level_labels),  index=pd.MultiIndex(levels=[['Actual:'], classes], labels=level_labels))
    return cm_frame

def display_classification_report(true_labels, predicted_labels, classes=[1,0]):

    report = metrics.classification_report(y_true=true_labels, y_pred=predicted_labels, labels=classes)
    print(report)



def display_model_performance_metrics(true_labels, predicted_labels, classes=[1,0]):
    print('Model Performance metrics:')
    print('-'*30)
    get_metrics(true_labels=true_labels, predicted_labels=predicted_labels)
    print('\nModel Classification report:')
    print('-'*30)
    display_classification_report(true_labels=true_labels, predicted_labels=predicted_labels, classes=classes)
    print('\nPrediction Confusion Matrix:')
    print('-'*30)
    print ('000')
    display_confusion_matrix(true_labels=true_labels, predicted_labels=predicted_labels, classes=classes)


def plot_model_decision_surface(clf, train_features, train_labels,
                                plot_step=0.02, cmap=plt.cm.RdYlBu,
                                markers=None, alphas=None, colors=None):

    if train_features.shape[1] != 2:
        raise ValueError("X_train should have exactly 2 columnns!")

    x_min, x_max = train_features[:, 0].min() - plot_step, train_features[:, 0].max() + plot_step
    y_min, y_max = train_features[:, 1].min() - plot_step, train_features[:, 1].max() + plot_step
    xx, yy = np.meshgrid(np.arange(x_min, x_max, plot_step),
                         np.arange(y_min, y_max, plot_step))

    clf_est = clone(clf)
    clf_est.fit(train_features,train_labels)
    if hasattr(clf_est, 'predict_proba'):
        Z = clf_est.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1]
    else:
        Z = clf_est.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    cs = plt.contourf(xx, yy, Z, cmap=cmap)

    le = LabelEncoder()
    y_enc = le.fit_transform(train_labels)
    n_classes = len(le.classes_)
    plot_colors = ''.join(colors) if colors else [None] * n_classes
    label_names = le.classes_
    markers = markers if markers else [None] * n_classes
    alphas = alphas if alphas else [None] * n_classes
    for i, color in zip(range(n_classes), plot_colors):
        idx = np.where(y_enc == i)
        plt.scatter(train_features[idx, 0], train_features[idx, 1], c=color,
                    label=label_names[i], cmap=cmap, edgecolors='black',
                    marker=markers[i], alpha=alphas[i])
    plt.legend()
    plt.show()


def plot_model_roc_curve(clf, features, true_labels, label_encoder=None, class_names=None):

    ## Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    if hasattr(clf, 'classes_'):
        class_labels = clf.classes_
    elif label_encoder:
        class_labels = label_encoder.classes_
    elif class_names:
        class_labels = class_names
    else:
        raise ValueError('Unable to derive prediction classes, please specify class_names!')
    n_classes = len(class_labels)
    y_test = label_binarize(true_labels, classes=class_labels)
    if n_classes == 2:
        if hasattr(clf, 'predict_proba'):
            prob = clf.predict_proba(features)
            y_score = prob[:, prob.shape[1]-1]
        elif hasattr(clf, 'decision_function'):
            prob = clf.decision_function(features)
            y_score = prob[:, prob.shape[1]-1]
        else:
            raise AttributeError("Estimator doesn't have a probability or confidence scoring system!")

        fpr, tpr, _ = roc_curve(y_test, y_score)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label='ROC curve (area = {0:0.2f})'
                                 ''.format(roc_auc),
                 linewidth=2.5)

    elif n_classes > 2:
        if hasattr(clf, 'predict_proba'):
            y_score = clf.predict_proba(features)
        elif hasattr(clf, 'decision_function'):
            y_score = clf.decision_function(features)
        else:
            raise AttributeError("Estimator doesn't have a probability or confidence scoring system!")

        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_test[:, i], y_score[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        ## Compute micro-average ROC curve and ROC area
        fpr["micro"], tpr["micro"], _ = roc_curve(y_test.ravel(), y_score.ravel())
        roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

        ## Compute macro-average ROC curve and ROC area
        # First aggregate all false positive rates
        all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
        # Then interpolate all ROC curves at this points
        mean_tpr = np.zeros_like(all_fpr)
        for i in range(n_classes):
            mean_tpr += interp(all_fpr, fpr[i], tpr[i])
        # Finally average it and compute AUC
        mean_tpr /= n_classes
        fpr["macro"] = all_fpr
        tpr["macro"] = mean_tpr
        roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

        ## Plot ROC curves
        plt.figure(figsize=(6, 4))
        plt.plot(fpr["micro"], tpr["micro"],
                 label='micro-average ROC curve (area = {0:0.2f})'
                       ''.format(roc_auc["micro"]), linewidth=3)

        plt.plot(fpr["macro"], tpr["macro"],
                 label='macro-average ROC curve (area = {0:0.2f})'
                       ''.format(roc_auc["macro"]), linewidth=3)

        for i, label in enumerate(class_labels):
            plt.plot(fpr[i], tpr[i], label='ROC curve of class {0} (area = {1:0.2f})'
                                           ''.format(label, roc_auc[i]),
                     linewidth=2, linestyle=':')
    else:
        raise ValueError('Number of classes should be atleast 2 or more')

    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    plt.show()

In [17]:
X_train = train_df["clean_text"]
y_train = train_df['label']
X_test = test_df["clean_text"]
y_test = test_df["label"]

In [18]:
# build TFIDF features on train reviews
tv = TfidfVectorizer(use_idf=True, min_df=0.0, max_df=1.0, ngram_range=(1,2),
                     sublinear_tf=True)
tv_train_features = tv.fit_transform(X_train)
tv_test_features = tv.transform(X_test)

In [19]:
from sklearn.linear_model import SGDClassifier, LogisticRegression

lr = LogisticRegression(penalty='l2', max_iter=100, C=1)
sgd = SGDClassifier(loss='hinge', n_iter_no_change=100)

In [20]:
# Logistic Regression model on tf-idf features
lr_tf_predictions = train_predict_model(classifier=lr,
                                             train_features=tv_train_features, train_labels=y_train,
                                             test_features=tv_test_features, test_labels=y_test)
display_model_performance_metrics(true_labels=y_test, predicted_labels=lr_tf_predictions,
                                      classes=[0,1,2])

c:\Code\HCAI\AI-ML\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Model Performance metrics:
------------------------------
Accuracy: 0.734
Precision: 0.7196
Recall: 0.734
F1 Score: 0.7216

Model Classification report:
------------------------------
              precision    recall  f1-score   support

           0       0.74      0.70      0.72       786
           1       0.49      0.28      0.36       239
           2       0.76      0.87      0.81       975

    accuracy                           0.73      2000
   macro avg       0.66      0.62      0.63      2000
weighted avg       0.72      0.73      0.72      2000


Prediction Confusion Matrix:
------------------------------
000
          Predicted:         
                   0   1    2
Actual: 0        552  40  194
        1        101  67   71
        2         95  31  849


In [21]:
# sgd Regression model on tf-idf features
sgd_tf_predictions = train_predict_model(classifier=sgd,
                                             train_features=tv_train_features, train_labels=y_train,
                                             test_features=tv_test_features, test_labels=y_test)
display_model_performance_metrics(true_labels=y_test, predicted_labels=sgd_tf_predictions,
                                      classes=[0,1,2])

Model Performance metrics:
------------------------------
Accuracy: 0.744
Precision: 0.735
Recall: 0.744
F1 Score: 0.7377

Model Classification report:
------------------------------
              precision    recall  f1-score   support

           0       0.75      0.73      0.74       786
           1       0.50      0.37      0.43       239
           2       0.78      0.85      0.81       975

    accuracy                           0.74      2000
   macro avg       0.68      0.65      0.66      2000
weighted avg       0.73      0.74      0.74      2000


Prediction Confusion Matrix:
------------------------------
000
          Predicted:         
                   0   1    2
Actual: 0        571  45  170
        1         91  89   59
        2        103  44  828


# Using lexical Models (Unsupervised learning)

In [22]:
!pip install Afinn

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for Afinn: filename=afinn-0.1-py3-none-any.whl size=53479 sha256=aa42a17998cbbb8199aa3c67825b819df4a0cf7189fd3c3c8d80edc3525f0abd
  Stored in directory: c:\users\navee\appdata\local\pip\cache\wheels\ee\d3\a0\f9255ebac29886acb1c28b35b37523f6399677fa06be379f25
Successfully built Afinn


In [23]:
from afinn import Afinn
afn = Afinn(emoticons=True)

In [24]:
test_reviews = test_df['clean_text'].tolist()

This dataset is designed for sentiment analysis tasks, providing labeled examples across three sentiment categories:

0: Negative
1: Neutral
2: Positive

In [25]:
sentiment_polarity = [afn.score(review) for review in test_reviews]
predicted_sentiments = [2 if score >= 1.0 else (0 if score <= -1.0 else 1) for score in sentiment_polarity]

In [26]:
display_model_performance_metrics(true_labels=y_test, predicted_labels=predicted_sentiments,
                                      classes=[0,1,2])

Model Performance metrics:
------------------------------
Accuracy: 0.5395
Precision: 0.6758
Recall: 0.5395
F1 Score: 0.5792

Model Classification report:
------------------------------
              precision    recall  f1-score   support

           0       0.76      0.45      0.57       786
           1       0.18      0.53      0.26       239
           2       0.73      0.61      0.66       975

    accuracy                           0.54      2000
   macro avg       0.56      0.53      0.50      2000
weighted avg       0.68      0.54      0.58      2000


Prediction Confusion Matrix:
------------------------------
000
          Predicted:          
                   0    1    2
Actual: 0        356  276  154
        1         43  126   70
        2         67  311  597


### By comparing two models we can see the supervised training gives better accuracy in every metrics than the lexical unsupervised models

## Notice that the text in the dataset is very short in some cases, so the accuracy of models may be relatively low.
Briefly discuss how you can potentially address this problem

Short texts (31% of the training sample have ≤5 words) give few features, so TF-IDF vectors are sparse and rare words are unseen. Remedies: lighter preprocessing, character n-grams, pretrained embeddings or transformers, and lexicon features as extra input.